# Forecast: sun strenth (gr_w: in W/m2)

In [2]:
import requests
import pandas as pd
from datetime import datetime, date, timedelta
import time

## Historische Metingen

In [18]:
# Basis configuratie
locatie = "Baarle-Nassau" ### KIES LOCATIE!
api_key = "1ad6efe54c"
base_url = "https://data.meteoserver.nl/api/historie.php"

# Definieer de periode (november 2024)
start_datum = date(2024, 9, 1)
eind_datum = date(2024, 9, 30)
delta = timedelta(days=1)

alle_data = []
huidige_datum = start_datum

while huidige_datum <= eind_datum:
    # Formatteer datum naar YYYYMMDD (bijv. 20241101)
    datum_str = huidige_datum.strftime('%Y%m%d')
    
    # Bouw de volledige URL op
    url = f"{base_url}?locatie={locatie}&dag={datum_str}&key={api_key}"
    
    print(f"Ophalen: {datum_str}...")
    r = requests.get(url)
    
    if r.status_code == 200:
        data = r.json()
        alle_data.append(data)
    else:
        print(f"Fout bij {datum_str}: Status {r.status_code}")

    # Belangrijk: Meteoserver heeft een limiet van 500 gratis requests per maand
    # Een kleine pauze voorkomt dat je de server overbelast
    time.sleep(0.5) 
    
    # Volgende dag
    huidige_datum += delta

Ophalen: 20240901...
Ophalen: 20240902...
Ophalen: 20240903...
Ophalen: 20240904...
Ophalen: 20240905...
Ophalen: 20240906...
Ophalen: 20240907...
Ophalen: 20240908...
Ophalen: 20240909...
Ophalen: 20240910...
Ophalen: 20240911...
Ophalen: 20240912...
Ophalen: 20240913...
Ophalen: 20240914...
Ophalen: 20240915...
Ophalen: 20240916...
Ophalen: 20240917...
Ophalen: 20240918...
Ophalen: 20240919...
Ophalen: 20240920...
Ophalen: 20240921...
Ophalen: 20240922...
Ophalen: 20240923...
Ophalen: 20240924...
Ophalen: 20240925...
Ophalen: 20240926...
Ophalen: 20240927...
Ophalen: 20240928...
Ophalen: 20240929...
Ophalen: 20240930...


In [19]:
# Flatten (normalize) the JSON file
metingen = pd.json_normalize(alle_data,record_path=['waarneming']) # for the main data (list)

#  drop unnecessary columns (+transform dt column to pd datetime)
metingen = metingen.drop(["windr", "winds", "windsu", "windstoot", "mintemp", "temp", "dauwp", "zonduur", "neerslduur", "neerslsom", "luchtd", "zicht", "bewolking", "rv", "code", "mist", "regen", "sneeuw", "onweer", "ijs"], axis='columns')
metingen.head()

,datum,uur,straling
0,20240901,1,0
1,20240901,2,0
2,20240901,3,0
3,20240901,4,0
4,20240901,5,0


In [20]:
# add timestamp instead of datum en uur
metingen['ts'] = pd.to_datetime(
    metingen['datum'].astype(str) + ' ' + (metingen['uur'].astype(int) - 1).astype(str), 
    format='%Y%m%d %H'
)

metingen = metingen.drop(["datum", "uur"], axis='columns') ## drop overbodige kolommen

In [21]:
metingen.to_csv(r"C:\Users\20204113\OneDrive - TU Eindhoven\2_Research\2_CoolAI\DATA\zonmetingen September 2024 (API)\zonmetingen(september_H2).csv", index = None)